### Tín hiệu mua: MACD cắt lên Signal Line và giá đóng cửa lớn hơn SMA va EMA50 lớn hơn EMA100 va gia du doan > gia thuc te
### Tín hiệu bán: MACD cắt xuống Signal Line và giá đóng cửa nhỏ hơn SMA va EMA50 nhỏ hơn EMA100 va gia du doan > gia thuc te

# Load data

In [1]:
import sys 
sys.path.append('../Common')
from CommonSSIDWH import CommonSSIDWH

# Process

In [2]:
import talib
import pandas as pd
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression
import redis
import numpy as np

symbol = 'VIC'
from_date = datetime.strptime('2025-01-01', '%Y-%m-%d')
to_date = datetime.strptime('2025-10-29', '%Y-%m-%d')

data = CommonSSIDWH.loaddataSSI_Ext(symbol, from_date, to_date, "1d")
data["Close"] = pd.to_numeric(data["Close"], errors='coerce')
data["Open"] = pd.to_numeric(data["Open"], errors='coerce')
data['macd'], data['macd_signal'], data['macd_hist'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
data['sma'] = talib.SMA(data['Close'], timeperiod=50)
data['ema50'] = talib.EMA(data['Close'], timeperiod=50)
data['ema100'] = talib.EMA(data['Close'], timeperiod=100)

# ML
data['macd'] = data['macd'].bfill().ffill()
data['sma'] = data['sma'].bfill().ffill()
# Features
features = data[['Close', 'macd', 'sma']].shift(1).bfill().ffill()
# Target
target = data['Close']

# Huấn luyện mô hình
model = LinearRegression()
model.fit(features, target)

# Dự đoán giá
data['predicted_close'] = model.predict(features)

data['Buy_Signal'] = (data['macd'] > data['macd_signal']) & (data['Close'] > data['sma']) & (data['ema50'] > data['ema100']) & (data['predicted_close'] > data['Close'])
data['Sell_Signal'] = (data['macd'] < data['macd_signal']) & (data['Close'] < data['sma']) & (data['ema50'] < data['ema100']) & (data['predicted_close'] < data['Close']) 
# data['Buy_Signal'] = True
last_record = data.iloc[-1]

# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
# Nếu có tín hiệu thì đẩy qua Redis
# Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line Buy_Signal  Sell_Signal
# Tạo kết nối Redis
r = redis.Redis(host='localhost', port=6379, db=0)
# Tạo hash key
hash_key = 'Buoi14.1_CK SSI_Xay dung chien luoc ML, MACD, MA'
last_record = data.iloc[-1]

print(last_record)

# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
	for field, value in last_record.to_dict().items():
		# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
		if isinstance(value, pd.Timestamp):
			value = value.isoformat()
		elif isinstance(value, (int, np.uint64)):
			value = str(value)
		r.hset(hash_key, field, value)  
		r.hset(hash_key, 'Symbol', symbol)
		r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
	print(last_record)   
else:
	print('Không có tín hiệu!')   

Datetime           2025-10-29 00:00:00
Open                            219000
High                            219000
Low                             211900
Close                           212000
Volume                         2739200
macd                      15460.813275
macd_signal               16013.501753
macd_hist                  -552.688478
sma                           163304.0
ema50                     170951.08638
ema100                   142156.833211
predicted_close          221790.897167
Buy_Signal                       False
Sell_Signal                      False
Name: 203, dtype: object
Không có tín hiệu!


# Rap vao Auto Trade

In [3]:
# 1 ngay chay 1 lan
# Lam o nha

# Backtest

In [4]:
data

,Datetime,Open,High,Low,Close,Volume,macd,macd_signal,macd_hist,sma,ema50,ema100,predicted_close,Buy_Signal,Sell_Signal
0,2025-01-02,40500,40550,40300,40550,1340500,42.941214,NaN,NaN,42230.0,NaN,NaN,40966.628476,False,False
1,2025-01-03,40550,40550,40150,40500,1729100,42.941214,NaN,NaN,42230.0,NaN,NaN,40966.628476,False,False
2,2025-01-06,40400,40500,40300,40500,1625900,42.941214,NaN,NaN,42230.0,NaN,NaN,40915.342788,False,False
3,2025-01-07,40300,40550,40200,40500,1965000,42.941214,NaN,NaN,42230.0,NaN,NaN,40915.342788,False,False
4,2025-01-08,40400,40550,40300,40500,925800,42.941214,NaN,NaN,42230.0,NaN,NaN,40915.342788,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199,2025-10-23,202900,215500,200000,215000,4761200,16053.177040,16049.814879,3.362160,155414.0,163098.795037,135986.649736,204361.854463,False,False
200,2025-10-24,215400,222900,215100,219000,3412900,16436.886140,16127.229132,309.657009,157474.0,165290.999154,137630.478454,216605.852690,False,False
201,2025-10-27,222900,226500,214000,214000,3889100,16151.337588,16132.050823,19.286765,159390.0,167201.156049,139142.746208,220631.023966,True,False
202,2025-10-28,213500,220600,199100,220100,5205500,16230.166071,16151.673872,78.492199,161428.0,169275.620518,140745.860144,215553.275358,False,False


In [5]:
import CommonBacktest

dataBacktest = CommonBacktest.CommonBacktest.backtest(data, 100000000, 100)

Ngày vào lệnh đầu tiên: 100
Tổng lợi nhuận: 95150000
Tổng giá trị tài khoản: 195150000
Lợi nhuận thị trường: 422.8113440197287%
Lợi nhuận chiến lược: 95.15%


In [6]:
dataBacktest.to_csv('dataBacktest.csv', index=False)